# Protein Structure Prediction Workflow: From Selection to AlphaFold

## Overview
This notebook walks through the complete workflow of predicting protein structures using AI-powered tools like AlphaFold. You'll learn how to:
1. Select and retrieve proteins from databases
2. Prepare protein sequences for analysis
3. Run structure prediction models
4. Analyze and visualize predicted structures
5. Use AI assistants for interpretation

---

## Learning Objectives
By the end of this notebook, you should be able to:
- [ ] Query protein databases for sequences of interest
- [ ] Prepare and validate protein sequences
- [ ] Understand how AlphaFold predicts 3D structures
- [ ] Interpret confidence scores and structure quality metrics
- [ ] Use AI models to extract biological insights from predictions
- [ ] Visualize and analyze predicted protein structures

---

## Workflow Overview

![Protein Structure Prediction Workflow](assets/images/workflow_diagram.png)

*Note: Replace 'workflow_diagram.png' with your figure showing the complete workflow*

### The Complete Pipeline:

```
STEP 1: Protein Selection
├── Choose biological target
├── Query UniProt or PDB database
└── Retrieve protein sequence
        ↓
STEP 2: Sequence Preparation
├── Validate sequence format
├── Remove gaps and errors
└── Check for known structures
        ↓
STEP 3: Structure Prediction (AlphaFold)
├── Submit to prediction model
├── AI model processes sequence
└── Generate 3D coordinates
        ↓
STEP 4: Analysis & Quality Assessment
├── Examine confidence scores (pLDDT)
├── Identify high-confidence regions
└── Compare with experimental data (if available)
        ↓
STEP 5: Interpretation & Insights
├── Use AI assistants for annotation
├── Extract functional insights
└── Plan experimental validation
```

## Step 1: Protein Selection & Retrieval

### What is Protein Selection?
The first step is identifying which protein to study. This involves:
- **Biological motivation**: What biological question do you want to answer?
- **Database queries**: Find your protein in public databases
- **Sequence retrieval**: Get the amino acid sequence

### Common Protein Databases
| Database | URL | Best For |
|----------|-----|----------|
| UniProt | https://www.uniprot.org/ | Protein sequences & annotations |
| PDB | https://www.rcsb.org/ | Experimental 3D structures |
| NCBI | https://www.ncbi.nlm.nih.gov/ | Genetic sequences |
| InterPro | https://www.ebi.ac.uk/interpro/ | Protein domains & families |
| AlphaFoldDB | https://alphafold.ebi.ac.uk/ | Pre-computed predictions |

### Example: Selecting a Protein
Let's say we want to study **Insulin** (human hormone):
- **UniProt ID**: P01308
- **Gene**: INS
- **Organism**: Homo sapiens
- **Function**: Regulates blood glucose

In [ ]:
# Step 1: Retrieve protein sequence from UniProt
import requests
from Bio import SeqIO
import pandas as pd

# Example: Fetch Insulin sequence from UniProt
uniprot_id = "P01308"  # Human Insulin

# Query UniProt API
url = f"https://www.uniprot.org/uniprot/{uniprot_id}.fasta"
response = requests.get(url)

if response.status_code == 200:
    fasta_content = response.text
    print("Protein Information:")
    print(fasta_content[:500])  # Print first 500 characters
    
    # Parse with Biopython
    from io import StringIO
    fasta_io = StringIO(fasta_content)
    record = next(SeqIO.parse(fasta_io, "fasta"))
    
    print(f"\nProtein ID: {record.id}")
    print(f"Sequence Length: {len(record.seq)} amino acids")
    print(f"First 50 amino acids: {str(record.seq)[:50]}")
else:
    print(f"Error retrieving sequence: {response.status_code}")
    # Fallback: Example insulin sequence
    insulin_seq = "MALWMRLLPLLALTALPWWAKEAVTPEEAOLQCGSQSLYQLENYCN"
    print(f"Using example sequence: {insulin_seq}")

## Step 2: Sequence Preparation & Validation

### Why Prepare Sequences?
Before structure prediction, sequences must be:
- **Clean**: Free of gaps and invalid characters
- **Validated**: Correct amino acid codes
- **Checked**: Compare with known structures if available

### Amino Acid Codes
| Code | Amino Acid | Code | Amino Acid |
|------|------------|------|------------|
| A | Alanine | M | Methionine |
| C | Cysteine | N | Asparagine |
| D | Aspartic acid | P | Proline |
| E | Glutamic acid | Q | Glutamine |
| F | Phenylalanine | R | Arginine |
| G | Glycine | S | Serine |
| H | Histidine | T | Threonine |
| I | Isoleucine | V | Valine |
| K | Lysine | W | Tryptophan |
| L | Leucine | Y | Tyrosine |

In [ ]:
# Step 2: Prepare and validate protein sequences

def validate_protein_sequence(seq):
    """
    Validate protein sequence for AlphaFold input.
    
    Args:
        seq (str): Protein sequence
    
    Returns:
        tuple: (is_valid, cleaned_sequence, validation_report)
    """
    # Standard amino acids
    valid_aas = set('ACDEFGHIKLMNPQRSTVWY')
    
    # Convert to uppercase
    seq_clean = seq.upper()
    
    # Remove common characters that shouldn't be there
    seq_clean = seq_clean.replace(' ', '').replace('\n', '').replace('-', '')
    
    # Check for invalid characters
    invalid_chars = set(seq_clean) - valid_aas
    
    report = {
        'original_length': len(seq),
        'cleaned_length': len(seq_clean),
        'has_invalid_chars': len(invalid_chars) > 0,
        'invalid_chars': list(invalid_chars),
        'composition': compute_aa_composition(seq_clean)
    }
    
    is_valid = len(invalid_chars) == 0 and len(seq_clean) > 10
    
    return is_valid, seq_clean, report

def compute_aa_composition(seq):
    """
    Compute amino acid composition percentages.
    """
    aa_counts = {}
    for aa in seq:
        aa_counts[aa] = aa_counts.get(aa, 0) + 1
    
    # Convert to percentages
    total = len(seq)
    return {aa: (count/total)*100 for aa, count in aa_counts.items()}

# Example: Validate insulin sequence
example_seq = "MALWMRLLPLLALTALPWWAKEAVTPEEAOLQCGSQSLYQLENYCN"

is_valid, cleaned_seq, report = validate_protein_sequence(example_seq)

print("=== Sequence Validation Report ===")
print(f"Valid: {is_valid}")
print(f"Original length: {report['original_length']} aa")
print(f"Cleaned length: {report['cleaned_length']} aa")
print(f"Invalid characters: {report['invalid_chars']}")
print(f"\nAmino Acid Composition:")
for aa, percent in sorted(report['composition'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {aa}: {percent:.1f}%")

## Step 3: Structure Prediction with AlphaFold

### What is AlphaFold?
AlphaFold is a deep learning AI model that predicts 3D protein structures from amino acid sequences.

**Key Features:**
- Trained on experimental structure databases (PDB)
- Takes amino acid sequence as input
- Outputs predicted 3D coordinates
- Provides confidence scores (pLDDT)

### Running Structure Prediction

**Option 1: Online servers (easiest)**
- [AlphaFold Server](https://alphafold.ebi.ac.uk/) - Free, official
- [ColabFold](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb) - Fast, GPU-accelerated

**Option 2: Local installation (more control)**
- Install AlphaFold2 locally with GPU support
- Process multiple sequences
- Integrate into pipelines

### Understanding Confidence Scores (pLDDT)
- **pLDDT** = predicted Locally Distance Difference Test
- Ranges from 0-100
- **90+**: Very high confidence
- **70-90**: High confidence
- **50-70**: Medium confidence (caution)
- **<50**: Low confidence (unreliable)

In [ ]:
# Step 3: Simulate AlphaFold structure prediction
# In real usage, you would use ColabFold or local AlphaFold installation

import numpy as np
import matplotlib.pyplot as plt

class MockAlphaFoldPrediction:
    """
    Simulated AlphaFold prediction for demonstration.
    In practice, use real AlphaFold2 or ColabFold.
    """
    def __init__(self, sequence):
        self.sequence = sequence
        self.length = len(sequence)
        
    def predict(self):
        """
        Generate simulated prediction results.
        """
        # Simulated pLDDT scores
        # Real predictions follow amino acid sequence
        plddt = np.random.normal(75, 15, self.length)
        plddt = np.clip(plddt, 0, 100)
        
        # Simulated 3D coordinates (random for demo)
        xyz_coords = np.random.randn(self.length, 3) * 10
        
        return {
            'sequence': self.sequence,
            'plddt_scores': plddt,
            'coordinates_xyz': xyz_coords,
            'model_quality': np.mean(plddt)
        }

# Example prediction
seq = "MALWMRLLPLLALTALPWWAKEAVTPEEAOLQCGSQSLYQLENYCN"
predictor = MockAlphaFoldPrediction(seq)
results = predictor.predict()

print(f"Prediction Complete!")
print(f"Sequence: {results['sequence']}")
print(f"Sequence Length: {len(results['sequence'])} amino acids")
print(f"Average pLDDT: {results['model_quality']:.1f}")
print(f"\nPer-residue confidence scores:")
print(f"Min pLDDT: {np.min(results['plddt_scores']):.1f}")
print(f"Max pLDDT: {np.max(results['plddt_scores']):.1f}")
print(f"Mean pLDDT: {np.mean(results['plddt_scores']):.1f}")

## Step 4: Analysis & Quality Assessment

### Key Metrics to Examine

1. **pLDDT Score Distribution**
   - Shows confidence across the sequence
   - Peaks indicate high-confidence regions
   - Valleys indicate uncertain/flexible regions

2. **Secondary Structure Elements**
   - Alpha helices (α-helices)
   - Beta sheets (β-sheets)
   - Coils and loops

3. **Comparison with Experimental Data**
   - If PDB structure available, compare RMSD
   - RMSD < 2Å usually indicates good prediction

4. **Protein Domains**
   - Identify functional regions
   - Compare with InterPro domains

In [ ]:
# Step 4: Analyze prediction quality

def analyze_prediction_quality(plddt_scores, threshold=70):
    """
    Analyze prediction quality metrics.
    """
    high_conf = np.sum(plddt_scores >= threshold)
    total = len(plddt_scores)
    confidence_fraction = high_conf / total
    
    return {
        'total_residues': total,
        'high_confidence_residues': high_conf,
        'high_confidence_fraction': confidence_fraction,
        'confidence_threshold': threshold,
        'mean_plddt': np.mean(plddt_scores),
        'std_plddt': np.std(plddt_scores),
        'confidence_category': categorize_confidence(np.mean(plddt_scores))
    }

def categorize_confidence(mean_plddt):
    """
    Categorize prediction confidence.
    """
    if mean_plddt >= 90:
        return "Very high confidence"
    elif mean_plddt >= 70:
        return "High confidence"
    elif mean_plddt >= 50:
        return "Medium confidence"
    else:
        return "Low confidence - use with caution"

# Analyze our example
quality_metrics = analyze_prediction_quality(results['plddt_scores'], threshold=70)

print("=== Prediction Quality Analysis ===")
print(f"Total residues: {quality_metrics['total_residues']}")
print(f"High confidence residues (pLDDT ≥ {quality_metrics['confidence_threshold']}): {quality_metrics['high_confidence_residues']} ({quality_metrics['high_confidence_fraction']*100:.1f}%)")
print(f"Mean pLDDT: {quality_metrics['mean_plddt']:.1f} ± {quality_metrics['std_plddt']:.1f}")
print(f"Confidence category: {quality_metrics['confidence_category']}")

# Visualize confidence scores
fig, ax = plt.subplots(figsize=(12, 4))
residue_nums = np.arange(1, len(results['plddt_scores']) + 1)
colors = results['plddt_scores']

scatter = ax.scatter(residue_nums, results['plddt_scores'], c=colors, cmap='RdYlGn', s=50, alpha=0.7)
ax.axhline(y=70, color='blue', linestyle='--', label='High confidence threshold')
ax.axhline(y=50, color='orange', linestyle='--', label='Medium confidence threshold')
ax.set_xlabel('Residue Position')
ax.set_ylabel('pLDDT Score')
ax.set_title('Prediction Confidence Across Protein Sequence')
ax.set_ylim(0, 105)
ax.legend()
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('pLDDT Score')
plt.tight_layout()
plt.show()

## Step 5: AI-Assisted Interpretation

### Using LLMs for Protein Analysis

AI assistants can help with:
1. **Sequence annotation** - What functional domains are present?
2. **Function prediction** - What is this protein likely to do?
3. **Disease relevance** - Are mutations in this protein associated with diseases?
4. **Experimental planning** - What experiments would validate this prediction?
5. **Literature context** - What do we know about similar proteins?

### Example Prompts for AI Assistants

In [ ]:
# Step 5: AI-assisted interpretation
# These are example prompts to use with LLMs like Claude or GPT

example_prompts = {
    "sequence_analysis": """
    I have a protein sequence: MALWMRLLPLLALTALPWWAKEAVTPEEAOLQCGSQSLYQLENYCN
    
    Please analyze this sequence and tell me:
    1. What type of protein might this be (hormone, enzyme, structural protein, etc.)?
    2. What functional domains do you recognize?
    3. What biological processes might it be involved in?
    """,
    
    "structure_interpretation": """
    I have an AlphaFold prediction for a protein with these characteristics:
    - Mean pLDDT confidence: 75
    - High confidence regions: residues 10-30 and 45-60
    - Low confidence regions: residues 1-9 and 31-44
    
    What does this confidence pattern tell me about the protein structure?
    """,
    
    "experimental_design": """
    I have an AlphaFold structure prediction for [protein name]. 
    What experiments would best validate this prediction?
    What mutations would be informative to test?
    """,
    
    "disease_relevance": """
    I'm studying mutations in [protein]. The AlphaFold prediction suggests
    these mutations destabilize a key alpha-helix region.
    What diseases or conditions might this protein be involved in?
    """
}

print("=== Example AI Prompts for Protein Analysis ===")
print()
for prompt_type, prompt_text in example_prompts.items():
    print(f"\n[{prompt_type.upper()}]")
    print(prompt_text.strip())
    print("-" * 60)

## Practical Exercise

### Your Task:
1. **Select a protein** of interest from UniProt
2. **Retrieve its sequence** using the methods above
3. **Validate the sequence** using the validation function
4. **Predict structure** using ColabFold or AlphaFold Server
5. **Analyze results** using the quality metrics
6. **Write AI prompts** to interpret your findings

### Suggested Proteins to Study:
- **P01308** - Human Insulin (hormone, 51 aa)
- **P69905** - Hemoglobin alpha (oxygen transport, 141 aa)
- **P12345** - Myoglobin (oxygen binding, 154 aa)
- **P00720** - Carbonic anhydrase (enzyme, 260 aa)

### Deliverables:
1. Cleaned sequence (FASTA format)
2. pLDDT score plot
3. Quality assessment report
4. 3 AI prompts for interpretation
5. Brief summary of findings

In [ ]:
# Exercise: Try it yourself!
# Uncomment and modify to try different proteins

def complete_workflow(uniprot_id, protein_name):
    """
    Complete workflow from protein selection to analysis.
    """
    print(f"\n{'='*60}")
    print(f"WORKFLOW: {protein_name} ({uniprot_id})")
    print(f"{'='*60}")
    
    # Step 1: Retrieve sequence
    print("\nStep 1: Retrieving sequence from UniProt...")
    # [Your code to fetch from UniProt]
    example_seqs = {
        "P01308": "MALWMRLLPLLALTALPWWAKEAVTPEEAOLQCGSQSLYQLENYCN",
        "P69905": "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
    }
    
    sequence = example_seqs.get(uniprot_id, "")
    if not sequence:
        print(f"  Sequence not found for {uniprot_id}")
        return
    
    # Step 2: Validate
    print("Step 2: Validating sequence...")
    is_valid, cleaned_seq, validation_report = validate_protein_sequence(sequence)
    print(f"  Valid: {is_valid}")
    print(f"  Length: {len(cleaned_seq)} aa")
    
    # Step 3: Predict (simulated)
    print("Step 3: Running structure prediction...")
    predictor = MockAlphaFoldPrediction(cleaned_seq)
    results = predictor.predict()
    
    # Step 4: Analyze
    print("Step 4: Analyzing prediction quality...")
    quality = analyze_prediction_quality(results['plddt_scores'])
    print(f"  Mean pLDDT: {quality['mean_plddt']:.1f}")
    print(f"  High confidence: {quality['high_confidence_fraction']*100:.1f}%")
    print(f"  Category: {quality['confidence_category']}")
    
    print("\n✓ Workflow complete!")
    print("  Next: Submit to AlphaFold Server or ColabFold for real predictions")

# Try the workflow
complete_workflow("P01308", "Human Insulin")
# complete_workflow("P69905", "Hemoglobin Alpha")  # Uncomment to try

## Resources & Further Learning

### Online Tools
- [AlphaFold Server](https://alphafold.ebi.ac.uk/) - Official predictions
- [ColabFold](https://colab.research.google.com/github/sokrypton/ColabFold/) - Fast notebook-based predictions
- [Mol*](https://molstar.org/) - 3D protein structure visualization
- [PyMOL](https://pymol.org/) - Advanced structure analysis

### Python Libraries
- `biopython` - Sequence analysis
- `py3Dmol` - Structure visualization in Jupyter
- `biotite` - Advanced bioinformatics
- `MDAnalysis` - Molecular dynamics analysis

### Key Papers
- Jumper et al. (2021) - AlphaFold2 structure prediction
- Evans et al. (2022) - Biological insights from AlphaFold
- Tunyasuvunakool et al. (2021) - AlphaFold predictions on scale

### Related Modules
- Module 01: Sequence Analysis
- Module 02: Protein Structure (full module)
- Module 04: Genomics & Variant Analysis

---

## Summary

In this notebook, you learned:
1. ✓ How to select and retrieve proteins from databases
2. ✓ How to prepare and validate sequences
3. ✓ How AlphaFold predicts 3D structures
4. ✓ How to interpret confidence scores and quality metrics
5. ✓ How to use AI assistants for biological interpretation
6. ✓ How to design experiments to validate predictions

**Next Steps:**
- Try this workflow on proteins of interest
- Combine with other analysis tools
- Share findings with AI assistants for deeper insights
- Validate predictions experimentally